<a href="https://colab.research.google.com/github/Joaoplims/sna_roblox_kg/blob/main/Roblox_KG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install requests

## SETUP

In [2]:
from google.colab import userdata

API_KEY = userdata.get('roblox_oauth')
print( API_KEY )
HEADERS = {
    "x-api-key": API_KEY,
    "Content-Type": "application/json"
}

Bo3UKVecGk6QtPtNkA5tb2DIT+PFVGIxoNpqERPlnIdzQp2RZXlKaGJHY2lPaUpTVXpJMU5pSXNJbXRwWkNJNkluTnBaeTB5TURJeExUQTNMVEV6VkRFNE9qVXhPalE1V2lJc0luUjVjQ0k2SWtwWFZDSjkuZXlKaGRXUWlPaUpTYjJKc2IzaEpiblJsY201aGJDSXNJbWx6Y3lJNklrTnNiM1ZrUVhWMGFHVnVkR2xqWVhScGIyNVRaWEoyYVdObElpd2lZbUZ6WlVGd2FVdGxlU0k2SWtKdk0xVkxWbVZqUjJzMlVYUlFkRTVyUVRWMFlqSkVTVlFyVUVaV1IwbDRiMDV3Y1VWU1VHeHVTV1I2VVhBeVVpSXNJbTkzYm1WeVNXUWlPaUl4TURJME5USXhPRGMwTnlJc0ltVjRjQ0k2TVRjNE1UUTVORGsxTml3aWFXRjBJam94TnpneE5Ea3hNelUyTENKdVltWWlPakUzT0RFME9URXpOVFo5LkRGV0VWVVh2N0dYdk5oWTVwbmFseW5wSjhHRUkwbzBoNEhsVDhobVdybFM2ZzZMWTF1M2gwSEZnRDZNY3VuUDN5XzhORS1GRWFHd3pEeDlYTVBEUWJiemRvMW90UEx1Z2t5WXNOVmZrRUpmV1JDNUwwcUp3VGN4dmdUYU1heldDcTMwRGFtWlBFNXVrTF85OHN6ZU9MWG5XOXBRMGV4cUtZelMzdFNTX0ZuNUIwcm5kb2NNeGltc3c5bTBONmhrQVBtdnBxckppb28zVDNCblkweUpCRWllT2YzSUdoM1JlTWcwYllvdE02b1pIek9GS096OElfZ1A3R0lKUnl4UDBaVWNxU0NxZGNzUHloem5UODkxYUpSRm40WTBZWHR4SEpsLVJNYnpRTDhST2RRZkxjcnlUVHIzdjZXVXgtZDZqVHFDU3ZzLXBHcERna1dONmJQVjc5Zw==


## Utilidades

In [3]:
def save_df_to_csv(df, filename="data_export.csv"):
    """
    Salva um DataFrame do pandas em um arquivo CSV.
    """
    try:
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"✅ Arquivo '{filename}' salvo com sucesso!")
    except Exception as e:
        print(f"❌ Erro ao salvar o arquivo: {e}")

# Exemplo de uso com os dados processados anteriormente:
# save_df_to_csv(df_users, "usuarios_roblox.csv")

## Função de Chamada para a API

In [4]:
import requests
import time
import random

def make_roblox_request(url, method="GET", params=None, data=None, max_retries=5, auto_paginate=False):
    """
    Função genérica para realizar chamadas à API do Roblox com suporte a Rate Limiting e Paginação.
    """
    all_data = []
    current_params = params.copy() if params else {}

    while True:
        retry_count = 0
        success = False

        while retry_count <= max_retries:
            try:
                response = requests.request(
                    method=method,
                    url=url,
                    headers=HEADERS,
                    params=current_params,
                    json=data
                )

                if response.status_code == 200:
                    res_json = response.json()

                    # Lógica de detecção: se houver uma chave 'data' e for uma lista, tratamos como paginável
                    if isinstance(res_json, dict) and 'data' in res_json and isinstance(res_json['data'], list):
                        page_items = res_json['data']
                        all_data.extend(page_items)

                        next_cursor = res_json.get('nextPageCursor')
                        if auto_paginate and next_cursor:
                            current_params['cursor'] = next_cursor
                            print(f"📄 Página processada. Buscando próxima página...")
                            success = True
                            break
                        else:
                            print(f"✅ Sucesso [{method}]: {url} (Total itens: {len(all_data)})")
                            return {"data": all_data}
                    else:
                        # Se não for uma lista paginável, retorna o JSON bruto (caso da Cloud API v2)
                        print(f"✅ Sucesso [{method}]: {url} (Objeto individual)")
                        return res_json

                elif response.status_code == 429:
                    retry_count += 1
                    wait_time = int(response.headers.get("retry-after", (2 ** retry_count) + random.uniform(0, 1)))
                    print(f"⚠️ Rate Limited (429). Aguardando {wait_time}s...")
                    time.sleep(wait_time)
                    continue
                else:
                    print(f"❌ Erro {response.status_code}: {response.text}")
                    return None
            except Exception as e:
                print(f"❌ Erro na requisição: {e}")
                return None

        if not success: break
    return None

## Teste chamadas


In [5]:
target_uid = 947838656
user_api_url = f"https://apis.roblox.com/cloud/v2/users/{target_uid}"

print(f"📡 Consultando dados para o usuário: {target_uid}...")
user_data_response = make_roblox_request(user_api_url)

import json
if user_data_response:
    print("✅ Resposta recebida:")
    print(json.dumps(user_data_response, indent=4))
else:
    print("❌ Não foi possível obter os dados do usuário.")

📡 Consultando dados para o usuário: 947838656...
✅ Sucesso [GET]: https://apis.roblox.com/cloud/v2/users/947838656 (Objeto individual)
✅ Resposta recebida:
{
    "path": "users/947838656",
    "createTime": "2019-01-28T20:56:39.770Z",
    "id": "947838656",
    "name": "Scriptbloxian",
    "displayName": "Scriptbloxian",
    "about": "Lead Developer of Scriptbloxian Studios\nhttps://www.roblox.com/groups/4705120/Scriptbloxian-Studios",
    "locale": "en_us",
    "premium": true
}


In [6]:
import requests
import json

# Teste direto sem a função make_roblox_request
test_uid = 947838656
test_url = f"https://apis.roblox.com/cloud/v2/users/{test_uid}"

print(f"📡 Teste Direto (requests): {test_url}")
test_response = requests.get(test_url, headers=HEADERS)

print(f"Status Code: {test_response.status_code}")
try:
    raw_json = test_response.json()
    print("Raw JSON Response:")
    print(json.dumps(raw_json, indent=4))
except Exception as e:
    print(f"Erro ao decodificar JSON: {e}")
    print(f"Conteúdo bruto: {test_response.text}")

📡 Teste Direto (requests): https://apis.roblox.com/cloud/v2/users/947838656
Status Code: 200
Raw JSON Response:
{
    "path": "users/947838656",
    "createTime": "2019-01-28T20:56:39.770Z",
    "id": "947838656",
    "name": "Scriptbloxian",
    "displayName": "Scriptbloxian",
    "about": "Lead Developer of Scriptbloxian Studios\nhttps://www.roblox.com/groups/4705120/Scriptbloxian-Studios",
    "locale": "en_us",
    "premium": true
}


## Encontrar ID Seed Group

In [7]:
import time
# Cache do id do grupo mais popular (Scriptbloxian Studios) para evitar chamadas excessiva
ID_SEED_GROUP = 0
# Buscar o Grupo 'Scriptbloxian Studios'
# Pesquisas indicam que é um dos grupos mais populares (https://roblox.fandom.com/pt-br/wiki/Scriptbloxian_Studios)
# Grupo de desenvolvedor de jogos famoso
group_search_url = "https://groups.roblox.com/v1/groups/search/lookup?groupName=Scriptbloxian%20Studios"
group_search_results = make_roblox_request(group_search_url)

if group_search_results and group_search_results.get('data'):
    group_info = group_search_results['data'][0]
    group_id = group_info['id']
    ID_SEED_GROUP = group_id
    print(f"🎯 Grupo Encontrado: {group_info['name']} (ID: {group_id})")
else:
    print("❌ Grupo não encontrado ou erro na busca.")

✅ Sucesso [GET]: https://groups.roblox.com/v1/groups/search/lookup?groupName=Scriptbloxian%20Studios (Total itens: 10)
🎯 Grupo Encontrado: Scriptbloxian Studios (ID: 4705120)


## Consulta de usuários

In [8]:
SAMPLE_SIZE = 25  # Parametrizável

# 2. Extrair membros do grupo semente
members_url = f"https://groups.roblox.com/v1/groups/{ID_SEED_GROUP}/users?sortOrder=Asc&limit={SAMPLE_SIZE}"
members_data = make_roblox_request(members_url, auto_paginate = False)

user_ids = []
if members_data and 'data' in members_data:
    user_ids = [member['user']['userId'] for member in members_data['data']]
    print(f"👥 Extraídos {len(user_ids)} usuários do grupo semente (ID: {ID_SEED_GROUP}).")
    print(f"Amostra de IDs: {user_ids}")
else:
    print("❌ Não foi possível extrair os membros do grupo.")

✅ Sucesso [GET]: https://groups.roblox.com/v1/groups/4705120/users?sortOrder=Asc&limit=25 (Total itens: 25)
👥 Extraídos 25 usuários do grupo semente (ID: 4705120).
Amostra de IDs: [947838656, 1027961385, 357829182, 244267273, 740396998, 731822913, 871184693, 262953699, 940534557, 970378427, 386824609, 747056849, 452079889, 519533783, 174791020, 363117394, 1041550312, 1038182669, 727787047, 557051148, 1023552809, 112492866, 363983769, 1014686408, 726703596]


### Processamento de Dados dos Usuários
Nesta etapa, consultamos a API de Usuários da Cloud API para obter detalhes como `createTime` e `locale`, calculando métricas derivadas.

In [9]:
from datetime import datetime, timezone
import pandas as pd
import time

processed_users = []
now = datetime.now(timezone.utc)

print(f"🔍 Processando {len(user_ids)} usuários...\n")

for uid in user_ids:
    url = f"https://apis.roblox.com/cloud/v2/users/{uid}"
    # A função corrigida agora retorna o objeto diretamente para a Cloud API v2
    user_data = make_roblox_request(url, auto_paginate=False)

    if user_data and user_data.get('createTime'):
        create_time_str = user_data['createTime']
        create_time_dt = datetime.fromisoformat(create_time_str.replace('Z', '+00:00'))

        delta = now - create_time_dt
        age = delta.days / 365.25

        processed_users.append({
            "user_id": user_data.get('id'),
            "display_name": user_data.get('displayName'),
            "created_at": create_time_str,
            "age_years": round(age, 2),
            "level": "Veterano" if age >= 10 else "Intermediário" if age >= 4 else "Iniciante"
        })
    else:
        print(f"⚠️ Falha ao obter detalhes do usuário {uid}")

    time.sleep(0.2)

df_users = pd.DataFrame(processed_users)
if not df_users.empty:
    display(df_users.head(10))
else:
    print("❌ Nenhum dado processado.")

🔍 Processando 25 usuários...

✅ Sucesso [GET]: https://apis.roblox.com/cloud/v2/users/947838656 (Objeto individual)
✅ Sucesso [GET]: https://apis.roblox.com/cloud/v2/users/1027961385 (Objeto individual)
✅ Sucesso [GET]: https://apis.roblox.com/cloud/v2/users/357829182 (Objeto individual)
✅ Sucesso [GET]: https://apis.roblox.com/cloud/v2/users/244267273 (Objeto individual)
✅ Sucesso [GET]: https://apis.roblox.com/cloud/v2/users/740396998 (Objeto individual)
✅ Sucesso [GET]: https://apis.roblox.com/cloud/v2/users/731822913 (Objeto individual)
✅ Sucesso [GET]: https://apis.roblox.com/cloud/v2/users/871184693 (Objeto individual)
✅ Sucesso [GET]: https://apis.roblox.com/cloud/v2/users/262953699 (Objeto individual)
✅ Sucesso [GET]: https://apis.roblox.com/cloud/v2/users/940534557 (Objeto individual)
✅ Sucesso [GET]: https://apis.roblox.com/cloud/v2/users/970378427 (Objeto individual)
✅ Sucesso [GET]: https://apis.roblox.com/cloud/v2/users/386824609 (Objeto individual)
✅ Sucesso [GET]: https:

,user_id,display_name,created_at,age_years,level
0,947838656,Scriptbloxian,2019-01-28T20:56:39.770Z,7.55,Intermediário
1,1027961385,Iamsmarterthanyahya3,2019-04-04T20:14:50.953Z,7.37,Intermediário
2,357829182,FallenGh,2017-08-02T17:39:17.880Z,9.04,Intermediário
3,244267273,kidsgivemesloppy,2017-02-20T05:17:52.727Z,9.49,Intermediário
4,740396998,Lloyd,2018-08-30T19:34:01.370Z,7.96,Intermediário
5,731822913,Kirill_pro300,2018-08-26T16:09:21.050Z,7.98,Intermediário
6,871184693,hunika201,2018-11-24T09:20:23.383Z,7.73,Intermediário
7,262953699,kevin2598,2017-03-14T15:33:27.687Z,9.43,Intermediário
8,940534557,furiouse_2,2019-01-23T01:56:45.160Z,7.56,Intermediário
9,970378427,Mid_Bacon,2019-02-16T16:54:47.510Z,7.50,Intermediário


### Exportação de Dados
Salvando o conjunto de dados processado em um arquivo CSV para uso externo.

In [ ]:
# Exportando os usuários processados
save_df_to_csv(df_users, "usuarios_roblox_processados.csv")

✅ Arquivo 'usuarios_roblox_processados.csv' salvo com sucesso!


## First Hop

### Mapeamento de Afiliações (First Hop)
Nesta etapa, buscamos todos os grupos de cada um dos usuários da nossa amostra inicial para descobrir as conexões entre eles.

In [10]:
graph_edges = []

print(f"🕸️ Iniciando mapeamento de afiliações para {len(user_ids)} usuários...")

for uid in user_ids:
    # Endpoint da API V1 para listar grupos/cargos do usuário
    affiliations_url = f"https://groups.roblox.com/v1/users/{uid}/groups/roles"

    # Como este endpoint retorna uma lista na chave 'data', nossa função make_roblox_request funcionará bem
    response = make_roblox_request(affiliations_url, auto_paginate=False)

    if response and 'data' in response:
        groups = response['data']
        for item in groups:
            group_info = item.get('group', {})
            role_info = item.get('role', {})

            # Registramos a aresta: Origem (User) -> Relação (Member) -> Destino (Group)
            edge = {
                "source_id": uid,
                "source_type": "User",
                "target_id": group_info.get('id'),
                "target_name": group_info.get('name'),
                "target_type": "Group",
                "role_name": role_info.get('name'),
                "role_rank": role_info.get('rank')
            }
            graph_edges.append(edge)

    # Pequeno delay para respeitar a API
    time.sleep(0.2)



# Criando um DataFrame para visualizar as conexões
df_graph = pd.DataFrame(graph_edges)
print(f"✅ Mapeamento concluído! Encontradas {len(df_graph)} conexões.")
display(df_graph.head(10))

🕸️ Iniciando mapeamento de afiliações para 25 usuários...
✅ Sucesso [GET]: https://groups.roblox.com/v1/users/947838656/groups/roles (Total itens: 9)
✅ Sucesso [GET]: https://groups.roblox.com/v1/users/1027961385/groups/roles (Total itens: 1)
✅ Sucesso [GET]: https://groups.roblox.com/v1/users/357829182/groups/roles (Total itens: 28)
✅ Sucesso [GET]: https://groups.roblox.com/v1/users/244267273/groups/roles (Total itens: 14)
✅ Sucesso [GET]: https://groups.roblox.com/v1/users/740396998/groups/roles (Total itens: 27)
✅ Sucesso [GET]: https://groups.roblox.com/v1/users/731822913/groups/roles (Total itens: 4)
✅ Sucesso [GET]: https://groups.roblox.com/v1/users/871184693/groups/roles (Total itens: 4)
✅ Sucesso [GET]: https://groups.roblox.com/v1/users/262953699/groups/roles (Total itens: 5)
✅ Sucesso [GET]: https://groups.roblox.com/v1/users/940534557/groups/roles (Total itens: 3)
✅ Sucesso [GET]: https://groups.roblox.com/v1/users/970378427/groups/roles (Total itens: 6)
✅ Sucesso [GET]: h

,source_id,source_type,target_id,target_name,target_type,role_name,role_rank
0,947838656,User,257737430,Mischief Mechanics,Group,Owner,255
1,947838656,User,10975897,XP Games.,Group,[D] Developer,253
2,947838656,User,32794027,Titan Trainers,Group,Admin,254
3,947838656,User,32073904,Hero Rivals Arena,Group,Owner,255
4,947838656,User,16345189,Super Slide,Group,Owner,255
5,947838656,User,16176597,Shinoro Studios,Group,Owner,255
6,947838656,User,16170756,Magnetworks,Group,Owner,255
7,947838656,User,11979111,Scriptbloxian Creation Group,Group,Owner,255
8,947838656,User,4705120,Scriptbloxian Studios,Group,Lead Developer,255
9,1027961385,User,4705120,Scriptbloxian Studios,Group,Legend,1


In [11]:
# Exibindo uma amostra das arestas criadas
if graph_edges:
    print("Amostra das conexões:")
    for edge in graph_edges[:35]:
        print(f"Usuário [{edge['source_id']}] participando do Grupo [{edge['target_name']}] como '{edge['role_name']}' (Rank: {edge['role_rank']})")
else:
    print("❌ Nenhuma aresta encontrada")

Amostra das conexões:
Usuário [947838656] participando do Grupo [Mischief Mechanics] como 'Owner' (Rank: 255)
Usuário [947838656] participando do Grupo [XP Games.] como '[D] Developer' (Rank: 253)
Usuário [947838656] participando do Grupo [Titan Trainers] como 'Admin' (Rank: 254)
Usuário [947838656] participando do Grupo [Hero Rivals Arena] como 'Owner' (Rank: 255)
Usuário [947838656] participando do Grupo [Super Slide] como 'Owner' (Rank: 255)
Usuário [947838656] participando do Grupo [Shinoro Studios] como 'Owner' (Rank: 255)
Usuário [947838656] participando do Grupo [Magnetworks] como 'Owner' (Rank: 255)
Usuário [947838656] participando do Grupo [Scriptbloxian Creation Group] como 'Owner' (Rank: 255)
Usuário [947838656] participando do Grupo [Scriptbloxian Studios] como 'Lead Developer' (Rank: 255)
Usuário [1027961385] participando do Grupo [Scriptbloxian Studios] como 'Legend' (Rank: 1)
Usuário [357829182] participando do Grupo [Krabby Krew] como 'Anchovies' (Rank: 1)
Usuário [3578

In [13]:
# Salvar o progresso do grafo
save_df_to_csv(df_graph, "conexoes_roblox_first_hop.csv")

✅ Arquivo 'conexoes_roblox_first_hop.csv' salvo com sucesso!


### Análise da Distribuição de Ranks
Nesta seção, visualizamos como os ranks (cargos) estão distribuídos nas conexões coletadas. Ranks mais altos (ex: 255) geralmente indicam donos ou administradores, enquanto ranks baixos (ex: 1) indicam membros comuns.

In [16]:
import pandas as pd

# Calculando a contagem absoluta
distribuicao = df_graph['role_rank'].value_counts().sort_index(ascending=False).reset_index()
distribuicao.columns = ['Role Rank', 'Quantidade']

# Calculando a porcentagem
distribuicao['Percentual (%)'] = (distribuicao['Quantidade'] / distribuicao['Quantidade'].sum() * 100).round(2)

# Exibindo a tabela completa
display(distribuicao)

,Role Rank,Quantidade,Percentual (%)
0,255,9,2.27
1,254,1,0.25
2,253,1,0.25
3,245,1,0.25
4,244,1,0.25
5,154,1,0.25
6,150,3,0.76
7,30,1,0.25
8,10,4,1.01
9,7,1,0.25


## Second Hop

In [17]:
# Extraindo os IDs de grupos únicos (target_id) do DataFrame de conexões
unique_group_ids = df_graph['target_id'].unique().tolist()

print(f"🔍 Total de conexões: {len(df_graph)}")
print(f"🎯 Total de grupos únicos identificados: {len(unique_group_ids)}")
print(f"Amostra dos IDs: {unique_group_ids[:10]}")

🔍 Total de conexões: 397
🎯 Total de grupos únicos identificados: 299
Amostra dos IDs: [257737430, 10975897, 32794027, 32073904, 16345189, 16176597, 16170756, 11979111, 4705120, 34990762]


### Coleta de Membros por Grupo (Second Hop)
Nesta etapa, para cada grupo mapeado anteriormente, solicitamos a lista de membros (limitada aos 20 primeiros) para construir a rede de conexões de segundo nível.

In [19]:
second_hop_edges = []


print(f"🚀 Iniciando coleta de membros para {len(unique_group_ids)} grupos...")

for i, group_id in enumerate(unique_group_ids):
    # Progresso a cada 20 grupos para não poluir o console
    if i % 20 == 0:
        print(f"📦 Processando grupo {i}/{len(unique_group_ids)}...")

    # Endpoint para listar membros do grupo
    members_url = f"https://groups.roblox.com/v1/groups/{group_id}/users?sortOrder=Asc&limit={SAMPLE_SIZE}"

    response = make_roblox_request(members_url, auto_paginate=False)

    if response and 'data' in response:
        for member in response['data']:
            user_info = member.get('user', {})
            role_info = member.get('role', {})

            second_hop_edges.append({
                "source_id": user_info.get('userId'),
                "source_name": user_info.get('username'),
                "source_type": "User",
                "target_id": group_id,
                "target_type": "Group",
                "role_name": role_info.get('name'),
                "role_rank": role_info.get('rank')
            })

    # Delay para evitar 429 (Rate Limit)
    time.sleep(0.1)

df_second_hop = pd.DataFrame(second_hop_edges)
print(f"\n✅ Coleta concluída!")
print(f"Total de novas conexões encontradas: {len(df_second_hop)}")
display(df_second_hop.head(10))

🚀 Iniciando coleta de membros para 299 grupos...
📦 Processando grupo 0/299...
❌ Erro 400: {"errors":[{"code":3,"message":"The user is invalid or does not exist.","userFacingMessage":"The user is invalid or does not exist."}]}
✅ Sucesso [GET]: https://groups.roblox.com/v1/groups/10975897/users?sortOrder=Asc&limit=25 (Total itens: 25)
✅ Sucesso [GET]: https://groups.roblox.com/v1/groups/32794027/users?sortOrder=Asc&limit=25 (Total itens: 25)
✅ Sucesso [GET]: https://groups.roblox.com/v1/groups/32073904/users?sortOrder=Asc&limit=25 (Total itens: 1)
✅ Sucesso [GET]: https://groups.roblox.com/v1/groups/16345189/users?sortOrder=Asc&limit=25 (Total itens: 4)
✅ Sucesso [GET]: https://groups.roblox.com/v1/groups/16176597/users?sortOrder=Asc&limit=25 (Total itens: 6)
✅ Sucesso [GET]: https://groups.roblox.com/v1/groups/16170756/users?sortOrder=Asc&limit=25 (Total itens: 1)
✅ Sucesso [GET]: https://groups.roblox.com/v1/groups/11979111/users?sortOrder=Asc&limit=25 (Total itens: 10)
✅ Sucesso [GET]

,source_id,source_name,source_type,target_id,target_type,role_name,role_rank
0,1711299025,PrincessRayn44,User,10975897,Group,[G] Gamer,1
1,2558399989,Izzybotyourgot,User,10975897,Group,[G] Gamer,1
2,2591680131,batmankian900,User,10975897,Group,[G] Gamer,1
3,2489593516,lonzell_9,User,10975897,Group,[G] Gamer,1
4,2335800629,WDGanns,User,10975897,Group,[G] Gamer,1
5,2560085628,joew5709,User,10975897,Group,[G] Gamer,1
6,2634340174,RICHIE1448,User,10975897,Group,[G] Gamer,1
7,14358363,xEmberz,User,10975897,Group,[G] Gamer,1
8,2339442932,Toenailskidnoob,User,10975897,Group,[G] Gamer,1
9,1469500187,dkdkjekdndn,User,10975897,Group,[G] Gamer,1


In [20]:
# Salvando o resultado do segundo salto
save_df_to_csv(df_second_hop, "conexoes_roblox_second_hop.csv")

✅ Arquivo 'conexoes_roblox_second_hop.csv' salvo com sucesso!


In [22]:
from google.colab import drive
import os

# 1. Montar o Drive para salvamento persistente
drive.mount('/content/drive')
save_path = '/content/drive/My Drive/roblox_graph_data/'
os.makedirs(save_path, exist_ok=True)

# 2. Remover duplicatas do Second Hop
df_second_hop = df_second_hop.drop_duplicates(subset=['source_id', 'target_id'])
print(f"✅ Duplicatas removidas. Total de conexões únicas: {len(df_second_hop)}")

# 3. Extrair usuários únicos do second hop para mapear suas outras afiliações
new_user_ids = df_second_hop['source_id'].unique().tolist()
second_hop_affiliations = []

print(f"🕸️ Mapeando afiliações para {len(new_user_ids)} novos usuários...")

for i, uid in enumerate(new_user_ids):
    affiliations_url = f"https://groups.roblox.com/v1/users/{uid}/groups/roles"
    response = make_roblox_request(affiliations_url, auto_paginate=False)

    if response and 'data' in response:
        for item in response['data']:
            group_info = item.get('group', {})
            role_info = item.get('role', {})
            second_hop_affiliations.append({
                "source_id": uid,
                "source_type": "User",
                "target_id": group_info.get('id'),
                "target_name": group_info.get('name'),
                "target_type": "Group",
                "role_name": role_info.get('name'),
                "role_rank": role_info.get('rank')
            })

    # 4. Salvar no Drive a cada 200 usuários
    if (i + 1) % 200 == 0:
        temp_df = pd.DataFrame(second_hop_affiliations)
        temp_df.to_csv(f"{save_path}checkpoint_batch_{i+1}.csv", index=False)
        print(f"💾 Checkpoint salvo: {i+1} usuários processados.")

    time.sleep(0.2)

# Resultado final do mapeamento expandido
df_expanded_graph = pd.DataFrame(second_hop_affiliations)
df_expanded_graph.to_csv(f"{save_path}full_second_hop_affiliations.csv", index=False)
print(f"✅ Mapeamento completo concluído! Total de arestas: {len(df_expanded_graph)}")
display(df_expanded_graph.head())

A saída de streaming foi truncada nas últimas 5000 linhas.
✅ Sucesso [GET]: https://groups.roblox.com/v1/users/18034215/groups/roles (Total itens: 6)
✅ Sucesso [GET]: https://groups.roblox.com/v1/users/18596359/groups/roles (Total itens: 5)
✅ Sucesso [GET]: https://groups.roblox.com/v1/users/15986689/groups/roles (Total itens: 3)
✅ Sucesso [GET]: https://groups.roblox.com/v1/users/104415244/groups/roles (Total itens: 95)
✅ Sucesso [GET]: https://groups.roblox.com/v1/users/163626872/groups/roles (Total itens: 4)
✅ Sucesso [GET]: https://groups.roblox.com/v1/users/173448199/groups/roles (Total itens: 3)
✅ Sucesso [GET]: https://groups.roblox.com/v1/users/173625430/groups/roles (Total itens: 4)
✅ Sucesso [GET]: https://groups.roblox.com/v1/users/167418632/groups/roles (Total itens: 3)
✅ Sucesso [GET]: https://groups.roblox.com/v1/users/170431153/groups/roles (Total itens: 5)
✅ Sucesso [GET]: https://groups.roblox.com/v1/users/169903686/groups/roles (Total itens: 4)
✅ Sucesso [GET]: https:

,source_id,source_type,target_id,target_name,target_type,role_name,role_rank
0,1711299025,User,13420266,PLANETERIUM,Group,⭐ Player,1
1,1711299025,User,35669818,크롱 스튜디오,Group,멤버,1
2,1711299025,User,393625465,Anime Slap Tower,Group,Member,1
3,1711299025,User,33092455,HAKI SHOP,Group,🦋Member,2
4,1711299025,User,16805444,Denis dave,Group,Member,1


## Montagem Knowladge Graph

In [26]:
import networkx as nx
import matplotlib.pyplot as plt

# 1. Instanciar o grafo direcionado
G = nx.DiGraph()

print(f"🏗️ Construindo grafo a partir de {len(df_expanded_graph)} conexões...")

# 2. Adicionar arestas e atributos usando itertuples (MUITO mais rápido)
for row in df_expanded_graph.itertuples(index=False):
    # Acessamos os valores usando notação de ponto em vez de colchetes
    user_node = f"U_{row.source_id}"
    group_node = f"G_{row.target_id}"

    # Adiciona nós com metadados de tipo
    G.add_node(user_node, type='User', label=row.source_id)
    G.add_node(group_node, type='Group', name=row.target_name, label=row.target_id)

    # Cria a aresta com atributos de Rank
    G.add_edge(
        user_node,
        group_node,
        role=row.role_name,
        rank=row.role_rank
    )

# 3. Métricas básicas do Grafo
num_nodes = G.number_of_nodes()
num_edges = G.number_of_edges()

print(f"✅ Grafo de Conhecimento montado!")
print(f"🔹 Total de Nós: {num_nodes}")
print(f"🔸 Total de Arestas: {num_edges}")

# Exibir os 5 grupos com maior Grau de Entrada (In-Degree)
# O in_degree revela exatamente quantos usuários da sua amostra estão naquele grupo
in_degree_dict = dict(G.in_degree())
sorted_in_degree = sorted(in_degree_dict.items(), key=lambda item: item[1], reverse=True)

print("\n🔝 Top 5 Grupos Hubs da Rede (Mais populares entre a amostra):")
for node, degree in sorted_in_degree[:5]:
    # Como filtramos por in_degree, os top serão inevitavelmente nós de 'Group' (G_)
    node_data = G.nodes[node]
    name = node_data.get('name', 'Unknown')
    print(f"- {node} ({name}): {degree} membros mapeados")

🏗️ Construindo grafo a partir de 266724 conexões...
✅ Grafo de Conhecimento montado!
🔹 Total de Nós: 72862
🔸 Total de Arestas: 266724

🔝 Top 5 Grupos Hubs da Rede (Mais populares entre a amostra):
- G_4705120 (Scriptbloxian Studios): 1265 membros mapeados
- G_3333298 (Rumble Studios): 1217 membros mapeados
- G_3982592 (Bee Swarm Simulator Club): 1190 membros mapeados
- G_3959677 (BIG Games Pets): 1132 membros mapeados
- G_3461453 (Nosniy Games): 1100 membros mapeados


### Análise de Influência (Centralidade)
Nesta etapa, calculamos o `degree_centrality` para identificar quais usuários possuem o maior número de conexões com diferentes grupos, indicando perfis altamente ativos ou influentes na rede mapeada.

In [27]:
# Calculando a centralidade de grau para os nós
centrality = nx.degree_centrality(G)

# Filtrando apenas nós do tipo 'User'
user_centrality = {
    node: score for node, score in centrality.items()
    if G.nodes[node].get('type') == 'User'
}

# Ordenando para encontrar os Top 10 usuários
sorted_users = sorted(user_centrality.items(), key=lambda x: x[1], reverse=True)[:10]

print("🏆 Top 10 Usuários mais conectados no Grafo:")
for node, score in sorted_users:
    user_id = G.nodes[node].get('label')
    # Tentando buscar o nome no dataframe original se disponível
    user_name = df_second_hop[df_second_hop['source_id'] == user_id]['source_name'].iloc[0] if user_id in df_second_hop['source_id'].values else "Unknown"
    print(f"- Usuário {user_id} ({user_name}): Score {score:.6f}")

🏆 Top 10 Usuários mais conectados no Grafo:
- Usuário 5420068 (HiddoDev): Score 0.002059
- Usuário 5080868749 (dbs_johnny): Score 0.002059
- Usuário 4166356547 (k2_nz): Score 0.002059
- Usuário 2308665054 (CodeMerge): Score 0.002059
- Usuário 3774652516 (Savitar1196): Score 0.002059
- Usuário 2607226698 (rice_fi78): Score 0.002059
- Usuário 3065793394 (jett273322): Score 0.002059
- Usuário 414281205 (Captainboy124): Score 0.002045
- Usuário 2235085140 (HudsonS1216): Score 0.002045
- Usuário 7126325238 (ikher501): Score 0.002045
